# 06. Retrieval Baseline

## 목표
- 코사인 유사도 Top-k 검색 (naive baseline)
- 메타데이터 필터링 (발주기관, 사업명)
- 프로젝트 가이드 질문 예시로 정성 평가
- 이후 고도화(Hybrid, Re-ranking 등)의 비교 기준선 확보

## 전략
고도화는 나중에. 먼저 **가장 단순한 검색이 얼마나 되는지** 측정한다.

> **Input**: `artifacts/chroma_db/`  
> **Output**: 검색 결과 (in-memory)  
> **Prev**: [05_embedding.ipynb](05_embedding.ipynb) | **Next**: [07_generation.ipynb](07_generation.ipynb)


In [1]:
import pandas as pd
import numpy as np
import time
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

load_dotenv()
openai_client = OpenAI()

In [2]:
ARTIFACTS_DIR = Path("../../artifacts")
CHROMA_DIR = ARTIFACTS_DIR / "chroma_db"
EMBEDDING_MODEL = "text-embedding-3-small"
COLLECTION_NAME = "rfp_chunks_v1"

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = chroma_client.get_collection(COLLECTION_NAME)
print(f"컬렉션 로딩: {collection.count():,}개 청크")

컬렉션 로딩: 13,951개 청크


## 1. 검색 함수 정의

In [3]:
def embed_query(query: str) -> list[float]:
    """쿼리를 임베딩 벡터로 변환한다."""
    response = openai_client.embeddings.create(input=query, model=EMBEDDING_MODEL)
    return response.data[0].embedding


def retrieve(
    query: str,
    k: int = 5,
    where: dict = None,
    where_document: dict = None,
) -> dict:
    """벡터 DB에서 쿼리와 유사한 청크를 검색한다."""
    start = time.time()
    query_embedding = embed_query(query)
    kwargs = {"query_embeddings": [query_embedding], "n_results": k}
    if where:
        kwargs["where"] = where
    if where_document:
        kwargs["where_document"] = where_document
    results = collection.query(**kwargs)
    elapsed = time.time() - start
    chunks = []
    for i in range(len(results['ids'][0])):
        chunks.append({
            'rank': i + 1,
            'score': round(1 - results['distances'][0][i], 4),
            'id': results['ids'][0][i],
            'text': results['documents'][0][i],
            'metadata': results['metadatas'][0][i],
        })
    return {'query': query, 'chunks': chunks, 'elapsed_ms': round(elapsed * 1000)}


def display_results(result: dict, text_limit: int = 100):
    """검색 결과를 DataFrame으로 표시한다."""
    rows = []
    for c in result['chunks']:
        m = c['metadata']
        rows.append({
            '순위': c['rank'],
            '유사도': c['score'],
            '사업명': m.get('사업명', '')[:20],
            '발주기관': m.get('발주기관', ''),
            '도메인': m.get('사업도메인', ''),
            '유형': m.get('content_type', ''),
            '내용': c['text'][:text_limit].replace('\n', ' '),
        })
    print(f"쿼리: {result['query']}")
    print(f"검색 시간: {result['elapsed_ms']}ms")
    display(pd.DataFrame(rows))


## 2. 기본 검색 테스트 (Top-k)

In [4]:
# 단순 의미 검색
display_results(retrieve("이러닝시스템 운영 요구사항", k=5))

쿼리: 이러닝시스템 운영 요구사항
검색 시간: 624ms


,순위,유사도,사업명,발주기관,도메인,유형,내용
0,1,0.5982,정읍체육트레이닝센터 통합운영관리시스템,전북특별자치도 정읍시,조달/계약,table,| 기능 요구사항 | SFR | 목표 시스템(사업)이 반드시 수행하여야 하거나 목표...
1,2,0.5975,(재공고)2024 조선대학교 SW중심,조선대학교,조달/계약,table,| | SFR-017 | 전공심화트랙별 이수과정 관리 | | | SFR-018 ...
2,3,0.5957,호계체육관 배드민턴장 및 탁구장 예약,경기도 안양시,조달/계약,table,| 요구사항 상세설명 | 정의 | 예약 시스템 개발을 위한 하드웨어 장비구성에 대한...
3,4,0.5938,수문자료정보관리시스템(HDIMS) 재,한국수자원조사기술원,공간정보/GIS,text,"| 요구사항 상세설명 | 정의 | 시스템 응답 속도 및 어플리케이션, 데이터 처리 ..."
4,5,0.5920,2024년 건설기술에 관한 특허·실용,한국발명진흥회 입찰공고,교통/물류,table,# 나. 성능 요구사항 | 요구사항 분류 | 요구사항 분류 | 성능 요구사항 | |...


In [5]:
display_results(retrieve("사업 예산 규모", k=5))

쿼리: 사업 예산 규모
검색 시간: 142ms


,순위,유사도,사업명,발주기관,도메인,유형,내용
0,1,0.4952,인천공항운영서비스㈜ 차세대 ERP시스,인천공항운영서비스(주),경영/행정,text,- 예산분류 및 예산분류코드 관리 - 사업예산 세목/비용코드관리 - 일반예산 세목/...
1,2,0.4942,인천공항운영서비스㈜ 차세대 ERP시스,인천공항운영서비스(주),경영/행정,text,상세설명 | 정의 | 예산에 대한 통제 및 집행 관리 | | | 세부내용 | ○ ...
2,3,0.4568,축산물이력관리시스템 개선(정보화 사업,축산물품질평가원,농축수산,table,# 일반현황 및 연혁 | 회사명 | | 대표자 | | | --- | --- ...
3,4,0.4451,전기안전 관제시스템 보안 모듈 개발,한국전기안전공사,안전/재난,table,# 사업관리자(PM) 이력사항 | 소 속 | 소 속 | | | 직 ...
4,5,0.4442,강릉어선안전조업국 상황관제시스템 구축,수협중앙회,안전/재난,text,# 세부명세서 □ 총사업비 : 일금 원 (￦ ) [① 개발비 : 일금 원 (￦ ...


## 3. 메타데이터 필터링 검색

발주기관이나 사업명으로 검색 범위를 좁힐 수 있습니다.
RFP 검색에서 사용자가 특정 기관/사업을 언급하는 경우 필수 기능입니다.

In [6]:
# 발주기관 필터
display_results(retrieve(
    "이러닝시스템 운영 요구사항",
    k=5,
    where={"발주기관": "국민연금공단"},
))

쿼리: 이러닝시스템 운영 요구사항
검색 시간: 233ms


,순위,유사도,사업명,발주기관,도메인,유형,내용
0,1,0.5541,2024년 이러닝시스템 운영 용역,국민연금공단,교육/학습,text,# 2-2. 이러닝시스템 운영 | 요구사항 분류 | 요구사항 분류 | 시스템 기...
1,2,0.5497,2024년 이러닝시스템 운영 용역,국민연금공단,교육/학습,text,# 2-5. 기타사항 | 요구사항 분류 | 요구사항 분류 | 프로젝트 관리 요구...
2,3,0.5396,사업장 사회보험료 지원 고시 개정에,국민연금공단,의료/바이오,table,# 4. 시스템 장비 구성 요구사항 | 요구사항 분류 | 요구사항 분류 | 시스...
3,4,0.5303,2024년 이러닝시스템 운영 용역,국민연금공단,교육/학습,text,# 2-4. 개인정보보호 및 정보보안 | 요구사항 분류 | 요구사항 분류 | 보...
4,5,0.5276,2024년 이러닝시스템 운영 용역,국민연금공단,교육/학습,text,# 기술능력 평가 기준 | 부문별 평가항목 | 부문별 평가항목 | 평가요소 | ...


In [7]:
# 사업명 키워드 필터 (문서 내용에 키워드 포함)
display_results(retrieve(
    "AI 기반 예측 요구사항",
    k=5,
    where_document={"$contains": "극저온"},
))

쿼리: AI 기반 예측 요구사항
검색 시간: 686ms


,순위,유사도,사업명,발주기관,도메인,유형,내용
0,1,0.4327,2025년도 중이온가속기용 극저온시스,기초과학연구원,기타 정보시스템,text,개정번호 : 0 발 행 일 : 2024. 10. 30 페 이 지 : 11/19 | ...
1,2,0.4240,2025년도 중이온가속기용 극저온시스,기초과학연구원,기타 정보시스템,text,개정번호 : 0 발 행 일 : 2024. 10. 30 페 이 지 : 14/19 | ...
2,3,0.4223,2025년도 중이온가속기용 극저온시스,기초과학연구원,기타 정보시스템,text,7 (연구원 양식 제공) 극저온시스템용 제어시스템(알람시스템 포함) 개선 8 연구원...
3,4,0.4048,2025년도 중이온가속기용 극저온시스,기초과학연구원,기타 정보시스템,text,자는 관리감독을 위해 운전 인력 외 현장대리인 1 인 이상을 극저온설비동 제어실에 ...
4,5,0.3983,2025년도 중이온가속기용 극저온시스,기초과학연구원,기타 정보시스템,text,현장에 비치된 비상연락망 체 1 특이사항 또는 위험(긴급)상황 발생 시 상황 전파 ...


In [8]:
# 보강된 메타데이터 필터 테스트
print("=== 기관유형 필터: 대학교 사업에서 학사시스템 검색 ===")
display_results(retrieve(
    "학사 정보시스템 요구사항",
    k=5,
    where={"기관유형": "대학교"},
))


=== 기관유형 필터: 대학교 사업에서 학사시스템 검색 ===
쿼리: 학사 정보시스템 요구사항
검색 시간: 178ms


,순위,유사도,사업명,발주기관,도메인,유형,내용
0,1,0.6403,(재공고)2024 조선대학교 SW중심,조선대학교,조달/계약,table,| | SFR-017 | 전공심화트랙별 이수과정 관리 | | | SFR-018 ...
1,2,0.6306,차세대 포털·학사 정보시스템 구축사업,고려대학교,교육/학습,text,요구 정의사항 상세 세부 설명 내용산출정보 | 요구사항 명칭 요구 정의사항 상세 세...
2,3,0.6164,전문대학 혁신지원사업 서영대학교 차세,서영대학교 산학협력단,교육/학습,table,| 요구사항 분류 | 요구사항 분류 | 성능 요구사항 | | --- | --- | ...
3,4,0.6073,차세대 포털·학사 정보시스템 구축사업,고려대학교,교육/학습,text,"등)으로 설치하여야 하며, 성능 및 안정성을 고려하여 필요할 경우 최신 버전이 아닌..."
4,5,0.6062,JST 공유대학(원) xAPI기반 L,전북대학교,교육/학습,text,# □ 요구사항 총괄표 | 구 분 | 설 명 | 개수 | | --- | --- ...


In [9]:
print("=== 사업도메인 필터: 교육/학습 도메인에서 검색 ===")
display_results(retrieve(
    "교육과정 운영 요구사항",
    k=5,
    where={"사업도메인": "교육/학습"},
))


=== 사업도메인 필터: 교육/학습 도메인에서 검색 ===
쿼리: 교육과정 운영 요구사항
검색 시간: 199ms


,순위,유사도,사업명,발주기관,도메인,유형,내용
0,1,0.6067,차세대 포털·학사 정보시스템 구축사업,고려대학교,교육/학습,text,| 요구사항 분류 | 요구사항 분류 | 기능 요구사항 | | 요구사항 번호 | 요구...
1,2,0.5940,전문대학 혁신지원사업 서영대학교 차세,서영대학교 산학협력단,교육/학습,text,| 요구사항 명칭 | 요구사항 명칭 | 교원업적 관리 | | 요구사항 상세설명 | ...
2,3,0.5842,2024년 이러닝시스템 운영 용역,국민연금공단,교육/학습,text,| 요구 사항 상세 설명 | 정의 | “직무지식진단”운영 | | | 세부 내용 |...
3,4,0.5835,국가교육과정정보센터(NCIC) 시스템,한국교육과정평가원,교육/학습,table,### 2. 상세 요구사항 ❍ 요구사항 총괄표 ### ❍ 요구사항 목록 ...
4,5,0.5783,2024년 이러닝시스템 운영 용역,국민연금공단,교육/학습,text,교육운영 등 ㆍ월별 또는 분기별 지급 ※ 상기 내용은 공단 내부사정에 따라 변경될 ...


In [10]:
print("=== 기술스택 필터: AI 기술이 포함된 사업에서 검색 ===")
display_results(retrieve(
    "인공지능 기반 예측 시스템 요구사항",
    k=5,
    where_document={"$contains": "AI"},
))


=== 기술스택 필터: AI 기술이 포함된 사업에서 검색 ===
쿼리: 인공지능 기반 예측 시스템 요구사항
검색 시간: 198ms


,순위,유사도,사업명,발주기관,도메인,유형,내용
0,1,0.5381,"(재공고, 협상) 서울 디지털성범죄",서울특별시 여성가족재단,AI/데이터,table,# 3. 제안요청 내용 ## 가. 요구사항 구성 | 요구사항 분류 | 설 명 (...
1,2,0.5372,"(재공고, 협상) 서울 디지털성범죄",서울특별시 여성가족재단,AI/데이터,table,# 3. 제안요청 내용 ## 가. 요구사항 구성 | 요구사항 분류 | 설 명 (...
2,3,0.5085,서울특별시교육청 지능정보화전략계획(I,서울특별시교육청,교육/학습,text,| --- | --- | --- | | 요구사항 고유번호 | 요구사항 고유번호 | ...
3,4,0.5044,서울특별시교육청 지능정보화전략계획(I,서울특별시교육청,교육/학습,text,## ‑ 기술 발전 추이를 감안한 단계적 인공지능 기술 적용 방안 마련 ## ‑...
4,5,0.4954,[재공고]차세대 통합정보시스템(ERP,한국가스공사,경영/행정,table,| | EAI/ API | ECR-018 | 신규 시스템 연계 솔루션 도입 | |...


## 4. 프로젝트 가이드 질문 예시 테스트

프로젝트 가이드에서 제시된 질문들로 검색 품질을 정성 평가합니다.

In [11]:
# Q1: 국민연금공단 이러닝시스템 사업 요구사항
display_results(retrieve(
    "국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘",
    k=5,
))

쿼리: 국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘
검색 시간: 313ms


,순위,유사도,사업명,발주기관,도메인,유형,내용
0,1,0.6124,2024년 이러닝시스템 운영 용역,국민연금공단,교육/학습,text,| | | 북러닝(교재포함) | 북러닝(교재포함) | 명 | 800 | | ...
1,2,0.6075,2024년 이러닝시스템 운영 용역,국민연금공단,교육/학습,text,붙임서류 1. 입찰참가자격을 증명하는 서류 사본 1부 2. 인감증명서 1부 3. 기...
2,3,0.6041,2024년 이러닝시스템 운영 용역,국민연금공단,교육/학습,table,# 기술제안서 요약서식 ### 【서식#2】 # 업체현황 및 연혁 | 회 사...
3,4,0.5958,2024년 이러닝시스템 운영 용역,국민연금공단,교육/학습,table,"# □ 정보보안, 개인정보보호 규정 준수 및 보안 관리 방안 제시 # □ 인수인계..."
4,5,0.5863,사업장 사회보험료 지원 고시 개정에,국민연금공단,의료/바이오,table,# 2024.3. ## 전화번호: 063-713-6482 | | 국민연금공단 ...


In [12]:
# Q2: 콘텐츠 개발 관리 요구사항 (후속 질문)
display_results(retrieve(
    "콘텐츠 개발 관리 요구 사항에 대해서 더 자세히 알려 줘",
    k=5,
    where={"발주기관": "국민연금공단"},
))

쿼리: 콘텐츠 개발 관리 요구 사항에 대해서 더 자세히 알려 줘
검색 시간: 151ms


,순위,유사도,사업명,발주기관,도메인,유형,내용
0,1,0.6216,2024년 이러닝시스템 운영 용역,국민연금공단,교육/학습,text,# 2-3. 콘텐츠 개발ㆍ관리 | 요구사항 분류 | 요구사항 분류 | 개발 요구...
1,2,0.6192,2024년 이러닝시스템 운영 용역,국민연금공단,교육/학습,text,"◦ 콘텐츠 관리를 위해 콘텐츠 원본 및 완성본 제공(인재개발부 1개, 해당부서 1개..."
2,3,0.5781,2024년 이러닝시스템 운영 용역,국민연금공단,교육/학습,text,"* 콘텐츠에서 사용되는 디자인, 이미지 등은 3개 이상 제안 ◦ 공단 네트워크 환경..."
3,4,0.5625,2024년 이러닝시스템 운영 용역,국민연금공단,교육/학습,text,| | 세부 내용 | ◦ 선정업체는 다음 각 호의 사항이 명시된 운영계획서를 제출...
4,5,0.5619,2024년 이러닝시스템 운영 용역,국민연금공단,교육/학습,text,"비정형 콘텐츠 ㆍ워라벨, 힐링, 여가 등의 모바일 전용 비정형콘텐츠 1,000개 이..."


In [13]:
# Q3: 기초과학연구원 극저온시스템 AI 예측
display_results(retrieve(
    "기초과학연구원 극저온시스템 사업에서 AI 기반 예측에 대한 요구사항이 있나",
    k=5,
))

쿼리: 기초과학연구원 극저온시스템 사업에서 AI 기반 예측에 대한 요구사항이 있나
검색 시간: 147ms


,순위,유사도,사업명,발주기관,도메인,유형,내용
0,1,0.5116,"(재공고, 협상) 서울 디지털성범죄",서울특별시 여성가족재단,AI/데이터,table,# 3. 제안요청 내용 ## 가. 요구사항 구성 | 요구사항 분류 | 설 명 (...
1,2,0.5088,[사전공개] 학업성취도 다차원 종단분,서울시립대학교,조달/계약,text,### - 인공지능 적용 개발을 위한 파일럿 테스트 진행(예: 중도이탈자에 대한 프...
2,3,0.5069,"(재공고, 협상) 서울 디지털성범죄",서울특별시 여성가족재단,AI/데이터,table,# 3. 제안요청 내용 ## 가. 요구사항 구성 | 요구사항 분류 | 설 명 (...
3,4,0.4883,2세대 전자조달시스템 기반구축사업,한국생산기술연구원,조달/계약,table,2세대 전자조달시스템 기반구축사업 # 2024. 4. 디지털행정추진실 한국생산...
4,5,0.4864,평택시 강소형 스마트시티 AI 기반의,케빈랩 주식회사,AI/데이터,table,# 요구사항 목록 | 요구사항 분류 | 요구사항ID | 요구사항명 | | --- |...


In [14]:
# Q4: 한국원자력연구원 선량 평가 시스템 사업 목적
display_results(retrieve(
    "한국 원자력 연구원에서 선량 평가 시스템 고도화 사업을 발주했는데 이 사업이 왜 추진되는지 목적을 알려 줘",
    k=5,
))

쿼리: 한국 원자력 연구원에서 선량 평가 시스템 고도화 사업을 발주했는데 이 사업이 왜 추진되는지 목적을 알려 줘
검색 시간: 140ms


,순위,유사도,사업명,발주기관,도메인,유형,내용
0,1,0.6899,한국원자력연구원 선량평가시스템 고도화,한국원자력연구원,안전/재난,text,1 사업 안내 # 1.1 사업 개요 ### 가. 사업명 : 한국원자력연구원 선...
1,2,0.6544,한국원자력연구원 선량평가시스템 고도화,한국원자력연구원,안전/재난,text,# 2.2. 정보화 현황 ## 2.2.1. 연구원 부서 시스템 현황 ○ 방사...
2,3,0.6498,한국원자력연구원 선량평가시스템 고도화,한국원자력연구원,안전/재난,table,# 3.3 추진일정 ○ 사업 기간 : 계약 체결일로부터 6개월 ○ 사업 추진 일...
3,4,0.6341,한국원자력연구원 선량평가시스템 고도화,한국원자력연구원,안전/재난,text,# 1.3. 사업 범위 ### 가. 평가장기(Organ) 개선 ○ 포트란 모...
4,5,0.6221,[사전공개] 학업성취도 다차원 종단분,서울시립대학교,조달/계약,text,# 정성적 지표의 평가기준 평가항목 및 배점 정성적 평가의 평가항목 및 기준은 ...


In [15]:
# Q5: 고려대 vs 광주과기원 비교
display_results(retrieve(
    "고려대학교 차세대 포털 시스템 사업이랑 광주과학기술원의 학사 시스템 기능개선 사업을 비교해 줄래",
    k=10,
))

쿼리: 고려대학교 차세대 포털 시스템 사업이랑 광주과학기술원의 학사 시스템 기능개선 사업을 비교해 줄래
검색 시간: 164ms


,순위,유사도,사업명,발주기관,도메인,유형,내용
0,1,0.6442,학사시스템 기능개선 사업,광주과학기술원,교육/학습,table,# 제안요청서 | 사업명 | 학사시스템 기능개선 사업 | | --- | --- ...
1,2,0.6125,차세대 포털·학사 정보시스템 구축사업,고려대학교,교육/학습,text,### 해 정보서비스의 품질을 강화하고 대학 교육 시스템의 경쟁력을 확보하고자 함 ...
2,3,0.6027,학사시스템 기능개선 사업,광주과학기술원,교육/학습,text,# 2024. 12. | 사업부서 | 정보운영팀 | 박민성 | 062-715-2...
3,4,0.6015,차세대 포털·학사 정보시스템 구축사업,고려대학교,교육/학습,text,| 평가 | 평가항목 | 평가기준 | 배점 | | --- | --- | --- | ...
4,5,0.5947,차세대 포털·학사 정보시스템 구축사업,고려대학교,교육/학습,text,# 사업 개요 1. 사업개요 ### 가. 사업명: 고려대학교 차세대 포털·학...
5,6,0.5930,차세대 포털·학사 정보시스템 구축사업,고려대학교,교육/학습,table,- 식별 및 인증 | | | | | | | | - 암호지원 | | | | ...
6,7,0.5897,차세대 포털·학사 정보시스템 구축사업,고려대학교,교육/학습,text,"※ 동점자 처리기준: ①기술평가점수가 높은자, ②배점이 큰 항목의 평가 점수가 높은..."
7,8,0.5875,차세대 포털·학사 정보시스템 구축사업,고려대학교,교육/학습,text,계획하였는지 평가 -각 단계마다 품질 요구사항을 점검하는 별도의 전문 인력이 투입되...
8,9,0.5848,차세대 포털·학사 정보시스템 구축사업,고려대학교,교육/학습,table,# 제안요청서 # 차세대포털·학사 정보시스템 구축 사업 고려대학교 2024. ...
9,10,0.5846,차세대 포털·학사 정보시스템 구축사업,고려대학교,교육/학습,table,# 제안요청서 # 차세대포털·학사 정보시스템 구축 사업 고려대학교 2024. ...


## 5. Top-k 값 비교 실험

k=3, 5, 10, 20에서 검색 결과가 어떻게 달라지는지 확인합니다.

In [16]:
test_query = "국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘"

topk_results = []
for k in [3, 5, 10, 20]:
    result = retrieve(test_query, k=k)
    scores = [c['score'] for c in result['chunks']]
    docs = set(c['metadata']['사업명'] for c in result['chunks'])
    correct = sum(1 for c in result['chunks'] if '이러닝' in c['metadata']['사업명'])
    topk_results.append({
        'k': k,
        '최고 유사도': max(scores),
        '최저 유사도': min(scores),
        '고유 문서 수': len(docs),
        '정답 문서 수': correct,
        '정밀도': f"{correct/k*100:.0f}%",
        '검색 시간(ms)': result['elapsed_ms'],
    })

display(pd.DataFrame(topk_results))

,k,최고 유사도,최저 유사도,고유 문서 수,정답 문서 수,정밀도,검색 시간(ms)
0,3,0.6124,0.6041,1,3,100%,165
1,5,0.6124,0.5863,2,4,80%,150
2,10,0.6124,0.5657,2,9,90%,133
3,20,0.6124,0.5509,6,14,70%,208


## 6. 검색 컨텍스트 조합 함수

Retrieval 결과를 LLM에 넘길 컨텍스트 문자열로 조합합니다.
07_generation에서 이 함수를 사용합니다.

In [17]:
def build_context(result: dict, max_tokens_est: int = 4000) -> str:
    """검색 결과를 LLM 컨텍스트 문자열로 조합한다.
    
    각 청크에 출처 정보를 붙여서 LLM이 근거를 인용할 수 있도록 한다.
    max_tokens_est 기준으로 청크를 추가하다가 초과하면 중단.
    """
    context_parts = []
    total_chars = 0
    max_chars = max_tokens_est * 2  # 한국어 ~0.5토큰/자 추정
    
    for c in result['chunks']:
        source = f"[출처: {c['metadata']['사업명']} | {c['metadata']['발주기관']}]"
        chunk_text = f"{source}\n{c['text']}"
        
        if total_chars + len(chunk_text) > max_chars:
            break
        
        context_parts.append(chunk_text)
        total_chars += len(chunk_text)
    
    return "\n\n---\n\n".join(context_parts)


# 테스트
result = retrieve("국민연금공단 이러닝시스템 사업 요구사항", k=5)
context = build_context(result)
print(f"컨텍스트 길이: {len(context):,}자 (청크 {len(result['chunks'])}개)")
print(f"\n--- 컨텍스트 앞부분 (500자) ---")
print(context[:500])

컨텍스트 길이: 3,486자 (청크 5개)

--- 컨텍스트 앞부분 (500자) ---
[출처: 2024년 이러닝시스템 운영 용역 | 국민연금공단]
|  |  | 북러닝(교재포함) | 북러닝(교재포함) | 명 | 800 |  |  |
|  |  | 전화외국어(교재포함) | 2개월 과정 | 명 | 110 |  |  |
|  |  |  | 3개월 과정 | 명 | 70 |  |  |  
### ※ 산출내역서 작성 시“제안 안내(p20~26)”를 반드시 참고하여 작성  
### ※ 부서별 필수교육 콘텐츠 운영단가는 부서별로 확정된 인원에 대한 공단 콘텐츠 운영이므로 직무교육 운영단가에 준함  
### ※ 외부콘텐츠․전문자격증 교육교재, 북러닝 및 전화외국어교육 등 실물 교재가 지급되는 항목은 반드시 실물 교재 제공에 배송료 포함 가격 제시  
### [별지 6]  
| 입찰참가신청서
※ 아래사항 중 해당되는 경우에만 기재하시기 바랍니다. | 입찰참가신청서
※ 아래사항 중 해당되는 경우에만 기재하시기 바랍니다. | 입찰참가신청서
※ 아래사항 중 해당되는 경우에만 기재하시기 바랍


## 7. Baseline 검색 성능 정성 평가

프로젝트 가이드 질문으로 현재 naive baseline의 강점과 약점을 정리합니다.

In [18]:
eval_questions = [
    {
        "query": "국민연금공단이 발주한 이러닝시스템 관련 사업 요구사항을 정리해 줘",
        "expected_doc": "이러닝시스템 운영 용역",
        "expected_agency": "국민연금공단",
        "type": "단일문서",
    },
    {
        "query": "기초과학연구원 극저온시스템 사업에서 AI 기반 예측에 대한 요구사항이 있나",
        "expected_doc": "극저온시스템 운전 용역",
        "expected_agency": "기초과학연구원",
        "type": "단일문서",
    },
    {
        "query": "한국 원자력 연구원에서 선량 평가 시스템 고도화 사업의 목적",
        "expected_doc": "선량 평가",
        "expected_agency": "한국원자력연구원",
        "type": "단일문서",
    },
    {
        "query": "고려대학교 차세대 포털 시스템 사업이랑 광주과학기술원의 학사 시스템 기능개선 사업을 비교",
        "expected_doc": "차세대 포털",
        "expected_agency": "고려대학교",
        "type": "다문서비교",
    },
]

eval_rows = []
for q in eval_questions:
    result = retrieve(q['query'], k=5)
    top1 = result['chunks'][0]
    top5_docs = [c['metadata']['사업명'] for c in result['chunks']]
    
    # 정답 문서가 Top-k에 포함되는지
    hit_at_1 = q['expected_doc'] in top1['metadata']['사업명']
    hit_at_5 = any(q['expected_doc'] in doc for doc in top5_docs)
    
    eval_rows.append({
        '질문 유형': q['type'],
        '기대 문서': q['expected_doc'][:15],
        'Top-1 정답': 'O' if hit_at_1 else 'X',
        'Top-5 포함': 'O' if hit_at_5 else 'X',
        'Top-1 유사도': top1['score'],
        'Top-1 실제문서': top1['metadata']['사업명'][:20],
        '검색시간(ms)': result['elapsed_ms'],
    })

eval_df = pd.DataFrame(eval_rows)
display(eval_df)

hit1 = sum(1 for r in eval_rows if r['Top-1 정답'] == 'O')
hit5 = sum(1 for r in eval_rows if r['Top-5 포함'] == 'O')
print(f"\nTop-1 정확도: {hit1}/{len(eval_rows)} ({hit1/len(eval_rows)*100:.0f}%)")
print(f"Top-5 Hit Rate: {hit5}/{len(eval_rows)} ({hit5/len(eval_rows)*100:.0f}%)")

,질문 유형,기대 문서,Top-1 정답,Top-5 포함,Top-1 유사도,Top-1 실제문서,검색시간(ms)
0,단일문서,이러닝시스템 운영 용역,O,O,0.6124,2024년 이러닝시스템 운영 용역,136
1,단일문서,극저온시스템 운전 용역,X,X,0.5116,"(재공고, 협상) 서울 디지털성범죄",145
2,단일문서,선량 평가,X,X,0.7063,한국원자력연구원 선량평가시스템 고도화,204
3,다문서비교,차세대 포털,X,O,0.6477,학사시스템 기능개선 사업,136



Top-1 정확도: 1/4 (25%)
Top-5 Hit Rate: 2/4 (50%)


## 8. 종합 요약 및 인사이트

### Baseline 검색 결과
- 코사인 유사도 Top-k + 메타데이터 필터링 구현
- 검색 시간: 쿼리당 ~100ms 수준

### 현재 baseline의 강점
- **특정 사업명을 직접 언급한 쿼리**: 메타 프리픽스 덕분에 잘 찾음
- **메타데이터 필터**: 발주기관/문서 키워드 필터 시 정확도 대폭 향상
- **속도**: 13,951개 청크에서 ~100ms로 실시간 응답 가능

### 현재 baseline의 약점
- **도메인 혼동**: '요구사항'같은 범용 표현에 다른 문서의 유사한 표 구조가 매칭
- **고유 키워드 검색**: '극저온', '선량' 같은 키워드가 벡터 유사도에서 잡히지 않을 수 있음
- **다문서 비교**: 두 문서를 동시에 찾아야 하는 쿼리에서 한쪽만 검색됨

### 고도화 방향 (추후)
| 방법 | 해결하는 문제 | 우선순위 |
|---|---|---|
| Hybrid Search (벡터+BM25) | 고유 키워드 누락 | 높음 |
| Re-ranking | 표 구조 유사도에 의한 오답 | 높음 |
| Multi-query | 다문서 비교 질문 | 중간 |
| MMR | 결과 다양성 부족 | 낮음 |

### 다음 단계
- `07_generation.ipynb`: LLM 답변 생성 + 프롬프트 최적화
- 현재 baseline 검색 결과를 컨텍스트로 사용